# WM-811K 웨이퍼맵 불량 패턴 분류 - 전체 실험 재현

로컬에 GPU가 없어서 실험은 Colab에서 진행.
데이터 다운로드부터 학습/평가까지 이 노트북 하나로 재현 가능.

- 런타임 > 런타임 유형 변경 > **GPU(T4)** 선택 후 실행
- 전체 실행 시간: 대략 15~20분 (데이터 다운로드 포함)
- 마지막 셀에서 결과 그림/지표를 zip으로 다운로드 → repo의 `results/`에 복사


In [ ]:
import kagglehub

# WM-811K 다운로드 (약 2GB, 공개 데이터셋이라 로그인 없이 받아짐)
path = kagglehub.dataset_download("qingyi/wm811k-wafer-map")
print(path)

import os
for f in os.listdir(path):
    print(f, os.path.getsize(os.path.join(path, f)) // 1024**2, "MB")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_pickle(os.path.join(path, "LSWMD.pkl"))
print(len(df))
df.head(3)


## 1. 라벨 정리 + EDA

failureType이 `[['Center']]` 같은 중첩 배열이라 문자열로 꺼내준다.
라벨 없는 행(빈 배열)이 63만 장 정도라 제외.


In [ ]:
CLASSES = ['none', 'Center', 'Donut', 'Edge-Loc', 'Edge-Ring',
           'Loc', 'Random', 'Scratch', 'Near-full']

def get_label(x):
    if isinstance(x, str):
        return x
    try:
        if len(x) == 0:
            return None
        return str(x[0][0])
    except (TypeError, IndexError):
        return None

df['label'] = df['failureType'].apply(get_label)
df = df[df['label'].isin(CLASSES)].reset_index(drop=True)
print(len(df), "labeled")
df['label'].value_counts()


In [ ]:
os.makedirs("results_export", exist_ok=True)

# 클래스 분포 (log scale 아니면 none 때문에 나머지가 안 보임)
counts = df['label'].value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.index, counts.values)
ax.set_yscale('log')
ax.set_ylabel('count (log)')
ax.set_title('WM-811K class distribution')
plt.xticks(rotation=30)
for i, v in enumerate(counts.values):
    ax.text(i, v, str(v), ha='center', va='bottom', fontsize=8)
fig.tight_layout()
fig.savefig("results_export/class_dist.png", dpi=150)
plt.show()


In [ ]:
# 클래스별 샘플 확인
fig, axes = plt.subplots(3, 9, figsize=(16, 6))
for ci, cls in enumerate(CLASSES):
    sub = df[df['label'] == cls].sample(3, random_state=0)
    for ri, (_, row) in enumerate(sub.iterrows()):
        ax = axes[ri, ci]
        ax.imshow(row['waferMap'], cmap='viridis')
        ax.axis('off')
        if ri == 0:
            ax.set_title(cls, fontsize=9)
fig.tight_layout()
fig.savefig("results_export/samples_per_class.png", dpi=150)
plt.show()


## 2. 전처리

- nearest 리사이즈로 64x64 통일 (값 0/1/2가 보간으로 깨지지 않게)
- none은 13,000장만 샘플링
- 70/15/15 stratified 분할

repo의 `src/prepare_data.py`와 동일한 로직.


In [ ]:
from skimage.transform import resize
from sklearn.model_selection import train_test_split

SIZE = 64
NONE_CAP = 13000
SEED = 42

rng = np.random.default_rng(SEED)
none_idx = df.index[df['label'] == 'none'].to_numpy()
keep_none = rng.choice(none_idx, NONE_CAP, replace=False)
drop = set(none_idx) - set(keep_none)
df2 = df.drop(index=list(drop)).reset_index(drop=True)
print(len(df2))

X = np.zeros((len(df2), SIZE, SIZE), dtype=np.uint8)
y = np.zeros(len(df2), dtype=np.int64)
for i, row in enumerate(df2.itertuples()):
    wm = resize(row.waferMap, (SIZE, SIZE), order=0,
                preserve_range=True, anti_aliasing=False)
    X[i] = wm.astype(np.uint8)
    y[i] = CLASSES.index(row.label)
    if i % 5000 == 0:
        print(i, end=' ')

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)
print()
print(len(y_tr), len(y_val), len(y_te))


## 3. 학습

repo의 `src/dataset.py`, `src/model.py`, `src/train.py`와 동일한 코드.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score

class WaferDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        wm = self.X[idx]
        if self.augment:
            k = np.random.randint(4)
            wm = np.rot90(wm, k)
            if np.random.rand() < 0.5:
                wm = np.fliplr(wm)
            wm = wm.copy()
        x = np.stack([(wm == 0), (wm == 1), (wm == 2)]).astype(np.float32)
        return torch.from_numpy(x), int(self.y[idx])

def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class WaferCNN(nn.Module):
    def __init__(self, num_classes=9, in_ch=3):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(in_ch, 32), conv_block(32, 64),
            conv_block(64, 128), conv_block(128, 128))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

sum(p.numel() for p in WaferCNN().parameters())


In [ ]:
import time, random

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    for xb, yb in loader:
        out = model(xb.to(device))
        preds.append(out.argmax(1).cpu().numpy())
        labels.append(yb.numpy())
    preds, labels = np.concatenate(preds), np.concatenate(labels)
    return (preds == labels).mean(), f1_score(labels, preds, average='macro'), preds

set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

train_ds = WaferDataset(X_tr, y_tr, augment=True)
val_ds = WaferDataset(X_val, y_val)

counts = np.bincount(y_tr, minlength=9).astype(np.float64)
sample_w = (1.0 / np.clip(counts, 1, None))[y_tr]
sampler = WeightedRandomSampler(torch.from_numpy(sample_w),
                                num_samples=len(y_tr), replacement=True)

train_loader = DataLoader(train_ds, batch_size=128, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=256, num_workers=2)

model = WaferCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_acc': [], 'val_f1': []}
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    val_acc, val_f1, _ = evaluate(model, val_loader, device)
    history['train_loss'].append(float(np.mean(losses)))
    history['val_acc'].append(float(val_acc))
    history['val_f1'].append(float(val_f1))

    mark = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')
        mark = ' *'
    print(f"[{epoch+1:2d}/{EPOCHS}] loss {np.mean(losses):.4f} | "
          f"val acc {val_acc:.4f} | val macro-F1 {val_f1:.4f} | "
          f"{time.time()-t0:.0f}s{mark}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history['train_loss'])
ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(history['val_acc'], label='val acc')
ax[1].plot(history['val_f1'], label='val macro-F1')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].set_title('validation')
fig.tight_layout()
fig.savefig("results_export/training_curve.png", dpi=150)
plt.show()


## 4. 테스트셋 평가


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load('best_model.pt'))
test_ds = WaferDataset(X_te, y_te)
test_loader = DataLoader(test_ds, batch_size=256)

test_acc, test_f1, preds = evaluate(model, test_loader, device)
print(f"test accuracy: {test_acc:.4f}")
print(f"test macro-F1: {test_f1:.4f}")
print()
report = classification_report(y_te, preds, target_names=CLASSES, digits=4)
print(report)
with open("results_export/metrics.txt", "w") as f:
    f.write(f"test accuracy: {test_acc:.4f}\ntest macro-F1: {test_f1:.4f}\n\n")
    f.write(report)


In [ ]:
cm = confusion_matrix(y_te, preds, normalize='true')
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(9):
    for j in range(9):
        if cm[i, j] >= 0.01:
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if cm[i, j] > 0.5 else 'black')
fig.colorbar(im)
fig.tight_layout()
fig.savefig("results_export/confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# 오분류 케이스 확인
wrong = np.where(preds != y_te)[0]
print(f"오분류 {len(wrong)}/{len(y_te)}")
pick = wrong[:16]
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax_, idx in zip(axes.flat, pick):
    ax_.imshow(X_te[idx], cmap='viridis')
    ax_.set_title(f"true {CLASSES[y_te[idx]]}\npred {CLASSES[preds[idx]]}",
                  fontsize=8)
    ax_.axis('off')
for ax_ in axes.flat[len(pick):]:
    ax_.axis('off')
fig.tight_layout()
fig.savefig("results_export/misclassified.png", dpi=150)
plt.show()


In [ ]:
# 결과물 다운로드
import shutil
shutil.make_archive("results_export", "zip", "results_export")

from google.colab import files
files.download("results_export.zip")


---

받은 zip 안의 그림들을 repo `results/`에 넣고,
`metrics.txt`의 accuracy / macro-F1을 README 결과 표에 기입하면 끝.
